# 02 — Preprocessing
**Loan Approval Prediction**

Goals:
- Handle missing values (median/mode imputation)
- Encode the target column (Approved → 1, Rejected → 0)
- Label-encode categorical features
- Split into train and test sets
- Apply StandardScaler
- Save preprocessed arrays and artifacts to disk

---

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # make src/ importable

import numpy as np
import pandas as pd
import pickle

from src.config import DATA_DIR, MODELS_DIR, TARGET_COL, SCALER_FILE, ENCODER_FILE
from src.data_loader import load_dataset, split_dataset
from src.preprocessing import full_pipeline, scale_features

print("Data directory  :", DATA_DIR)
print("Models directory:", MODELS_DIR)

Data directory  : /mnt/c/Users/Mega-PC/Desktop/stage Elevvo/Task_4_Loan_Approval_Prediction/data
Models directory: /mnt/c/Users/Mega-PC/Desktop/stage Elevvo/Task_4_Loan_Approval_Prediction/models


## 1. Load Raw Data

In [2]:
df = load_dataset()
display(df.head())

Dataset  :  4269 rows  |  13 columns
Columns  : ['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


## 2. Missing Value Imputation

In [3]:
from src.preprocessing import handle_missing_values

df_clean = handle_missing_values(df)
print("Missing values after imputation:")
print(df_clean.isnull().sum())

No missing values detected — skipping imputation.
Missing values after imputation:
loan_id                     0
no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64


## 3. Feature Encoding

In [4]:
from src.preprocessing import encode_target, encode_features, get_features_and_target

df_encoded = encode_target(df_clean)
df_encoded, encoders = encode_features(df_encoded, fit=True)

print("\nEncoded dataframe sample:")
display(df_encoded.head())

  Encoded 'education': {'Graduate': np.int64(0), 'Not Graduate': np.int64(1)}
  Encoded 'self_employed': {'No': np.int64(0), 'Yes': np.int64(1)}

Encoded dataframe sample:


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,0,0,9600000,29900000,12,778,2400000,17600000,22700000,8000000,1
1,2,0,1,1,4100000,12200000,8,417,2700000,2200000,8800000,3300000,0
2,3,3,0,0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,0
3,4,3,0,0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,0
4,5,5,1,1,9800000,24200000,20,382,12400000,8200000,29400000,5000000,0


## 4. Train / Test Split

In [5]:
X, y = get_features_and_target(df_encoded)
print(f"Features shape : {X.shape}")
print(f"Target  shape  : {y.shape}")
print(f"Feature columns: {list(X.columns)}")

X_train, X_test, y_train, y_test = split_dataset(X, y)
print(f"\nClass distribution in train: {dict(y_train.value_counts())}")
print(f"Class distribution in test : {dict(y_test.value_counts())}")

Features shape : (4269, 11)
Target  shape  : (4269,)
Feature columns: ['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']
Train    :  3415 samples  (approved=2125, rejected=1290)
Test     :   854 samples  (approved=531, rejected=323)

Class distribution in train: {1: np.int64(2125), 0: np.int64(1290)}
Class distribution in test : {1: np.int64(531), 0: np.int64(323)}


## 5. Feature Scaling

In [6]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test, fit=True)
print(f"Train scaled shape : {X_train_scaled.shape}")
print(f"Test  scaled shape : {X_test_scaled.shape}")

Train scaled shape : (3415, 11)
Test  scaled shape : (854, 11)


## 6. Save Artifacts

In [7]:
import pickle

os.makedirs(MODELS_DIR, exist_ok=True)

# Save scaler and label encoders
with open(SCALER_FILE, 'wb') as f:
    pickle.dump(scaler, f)

with open(ENCODER_FILE, 'wb') as f:
    pickle.dump(encoders, f)

# Save preprocessed arrays to data/
import numpy as np
np.save(os.path.join(DATA_DIR, 'X_train.npy'), X_train_scaled)
np.save(os.path.join(DATA_DIR, 'X_test.npy'),  X_test_scaled)
np.save(os.path.join(DATA_DIR, 'y_train.npy'), y_train.values)
np.save(os.path.join(DATA_DIR, 'y_test.npy'),  y_test.values)

# Save feature names for later use
feature_names = list(X.columns)
with open(os.path.join(DATA_DIR, 'feature_names.pkl'), 'wb') as f:
    pickle.dump(feature_names, f)

print("Artifacts saved:")
print(f"  Scaler   → {SCALER_FILE}")
print(f"  Encoders → {ENCODER_FILE}")
print(f"  Arrays   → data/X_train.npy, X_test.npy, y_train.npy, y_test.npy")
print(f"  Features → data/feature_names.pkl")

Artifacts saved:
  Scaler   → /mnt/c/Users/Mega-PC/Desktop/stage Elevvo/Task_4_Loan_Approval_Prediction/models/scaler.pkl
  Encoders → /mnt/c/Users/Mega-PC/Desktop/stage Elevvo/Task_4_Loan_Approval_Prediction/models/encoders.pkl
  Arrays   → data/X_train.npy, X_test.npy, y_train.npy, y_test.npy
  Features → data/feature_names.pkl


## 7. Scaling Verification

In [8]:
import pandas as pd

scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names)
print("Scaled training feature statistics:")
display(scaled_df.describe().round(3))

Scaled training feature statistics:


,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
count,3415.000,3415.000,3415.000,3415.000,3415.000,3415.000,3415.000,3415.000,3415.000,3415.000,3415.000
mean,-0.000,-0.000,0.000,0.000,-0.000,0.000,-0.000,0.000,0.000,-0.000,0.000
std,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
min,-1.468,-0.999,-1.009,-1.740,-1.657,-1.549,-1.732,-1.170,-1.137,-1.633,-1.541
25%,-0.878,-0.999,-1.009,-0.845,-0.817,-0.851,-0.864,-0.812,-0.840,-0.840,-0.799
50%,0.301,-0.999,0.992,0.015,-0.055,-0.153,-0.008,-0.282,-0.290,-0.059,-0.149
75%,0.891,1.001,0.992,0.838,0.696,0.894,0.865,0.591,0.603,0.711,0.623
max,1.480,1.001,0.992,1.734,2.656,1.592,1.738,3.379,3.305,2.647,3.004
